In [33]:
# !pip install catboost lightgbm xgboost

In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder, FunctionTransformer

from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/rps_data.csv', sep=',')
df['next_opp_move'] = df.groupby('match_id')['opp_move'].shift(-1)
df = df[df['next_opp_move'].notna()]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
df

,match_id,round,player_name,win_category,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,is_last_round,...,score_me_before,score_opp_before,streak_draws,prev_opp_move,prev_my_move,prev_outcome,prev2_opp_move,prev2_my_move,prev2_outcome,next_opp_move
0,1,1,Неизвестен,unknown,-1,-0.01,25,Н,Б,0,...,0,0,0,-1,-1,none,-1,-1,none,К
1,1,2,Неизвестен,unknown,-1,-0.01,25,К,Н,0,...,0,1,0,Н,Б,lose,-1,-1,none,Н
2,1,3,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,0,К,Н,lose,Н,Б,lose,Н
3,1,4,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,1,Н,Н,draw,К,Н,lose,Н
4,1,5,Неизвестен,unknown,-1,-0.01,25,Н,К,1,...,0,2,2,Н,Н,draw,Н,Н,draw,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,188,5,Дальний Тайник,<=5,3,0.75,100,Н,Н,1,...,1,2,0,К,Н,lose,Н,К,win,Б
1199,188,6,Дальний Тайник,<=5,3,0.75,100,Б,Б,1,...,1,2,1,Н,Н,draw,К,Н,lose,Н
1201,189,1,Дальний Тайник,<=5,4,0.80,50,Н,Б,0,...,0,0,0,-1,-1,none,-1,-1,none,К
1202,189,2,Дальний Тайник,<=5,4,0.80,50,К,Н,0,...,0,1,0,Н,Б,lose,-1,-1,none,К


In [36]:
# Первый раунд
def get_move_r1(stake):
    matches = len(df[(df["round"] == 1) & (df["stake"] == stake)])
    counts = df[(df["round"] == 1) & (df["stake"] == stake)]["opp_move"].value_counts()
    k_count = counts['К']
    b_count = counts['Б']
    n_count = counts['Н']
    p_k = k_count / matches
    p_b = b_count / matches
    p_n = n_count / matches
    exp_k = p_n - p_b
    exp_n = p_b - p_k
    exp_b = p_k - p_n
    if exp_k >= exp_n and exp_k >= exp_b:
        return 'К'
    elif exp_n >= exp_b:
        return 'Н'
    else:
        return 'Б'

# Второй раунд
df_r2 = df[df['round'] == 2].copy()
prob_r2 = df_r2.groupby(['stake', 'prev_outcome', 'prev_my_move', 'prev_opp_move'])['opp_move'].value_counts(normalize=True).reset_index(name='prob')

def get_move_r2(stake, outcome_r1, my_move_r1, opp_move_r1, prob_r2):
    """
    Возвращает оптимальный ход во втором раунде на основе статистики первого раунда.

    Параметры:
        stake: int - ставка в текущем матче
        outcome_r1: str - исход первого раунда ('win', 'loss', 'draw')
        my_move_r1: str - твой ход в первом раунде ('К', 'Н', 'Б')
        opp_move_r1: str - ход соперника в первом раунде ('К', 'Н', 'Б')
        prob_r2: DataFrame - таблица вероятностей (колонки: stake, prev_outcome, prev_my_move, prev_opp_move, opp_move, prob)

    Возвращает:
        str: 'К', 'Н' или 'Б'
    """
    # Ищем вероятности для данного контекста
    mask = (prob_r2['stake'] == stake) & \
           (prob_r2['prev_outcome'] == outcome_r1) & \
           (prob_r2['prev_my_move'] == my_move_r1) & \
           (prob_r2['prev_opp_move'] == opp_move_r1)

    subset = prob_r2[mask]

    if len(subset) == 0:
        # Нет данных для этого контекста
        return None

    # Извлекаем вероятности для каждого хода соперника
    p_k = subset[subset['opp_move'] == 'К']['prob'].sum()
    p_n = subset[subset['opp_move'] == 'Н']['prob'].sum()
    p_b = subset[subset['opp_move'] == 'Б']['prob'].sum()
    exp_k = p_n - p_b
    exp_n = p_b - p_k
    exp_b = p_k - p_n
    if exp_k >= exp_n and exp_k >= exp_b:
        return 'К'
    elif exp_n >= exp_b:
        return 'Н'
    else:
        return 'Б'

# Третий раунд
df_r3 = df[df['round'] == 3].copy()
prob_r3 = df_r3.groupby(['stake', 'prev_outcome', 'prev_my_move', 'prev_opp_move',
                         'prev2_outcome', 'prev2_my_move', 'prev2_opp_move'])['opp_move'].value_counts(normalize=True).reset_index(name='prob')

def get_move_r3(stake, outcome_r2, my_move_r2, opp_move_r2,
                       outcome_r1, my_move_r1, opp_move_r1, prob_r3):
    """
    Возвращает оптимальный ход в третьем раунде на основе статистики первых двух раундов.
    Если данных для контекста нет, возвращает None.

    Параметры:
        stake: int
        outcome_r2: str - исход второго раунда ('win','loss','draw')
        my_move_r2: str - твой ход во втором раунде
        opp_move_r2: str - ход соперника во втором раунде
        outcome_r1: str - исход первого раунда
        my_move_r1: str - твой ход в первом раунде
        opp_move_r1: str - ход соперника в первом раунде
        prob_r3: DataFrame - таблица вероятностей для третьего раунда

    Возвращает:
        str: 'К', 'Н', 'Б' или None
    """
    mask = (prob_r3['stake'] == stake) & \
           (prob_r3['prev_outcome'] == outcome_r2) & \
           (prob_r3['prev_my_move'] == my_move_r2) & \
           (prob_r3['prev_opp_move'] == opp_move_r2) & \
           (prob_r3['prev2_outcome'] == outcome_r1) & \
           (prob_r3['prev2_my_move'] == my_move_r1) & \
           (prob_r3['prev2_opp_move'] == opp_move_r1)

    subset = prob_r3[mask]

    if len(subset) == 0:
        return None

    # Суммируем вероятности для каждого хода соперника
    p_k = subset[subset['opp_move'] == 'К']['prob'].sum()
    p_n = subset[subset['opp_move'] == 'Н']['prob'].sum()
    p_b = subset[subset['opp_move'] == 'Б']['prob'].sum()

    # Ожидаемые выигрыши
    exp_k = p_n - p_b
    exp_n = p_b - p_k
    exp_b = p_k - p_n

    if exp_k >= exp_n and exp_k >= exp_b:
        return 'К'
    elif exp_n >= exp_b:
        return 'Н'
    else:
        return 'Б'

In [37]:
df_ml = df[df['round'] > 2].copy()

In [48]:
joblib.dump(preprocessor, 'rps_preprocessor.pkl')

['rps_preprocessor.pkl']

In [38]:
df_ml

,match_id,round,player_name,win_category,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,is_last_round,...,score_me_before,score_opp_before,streak_draws,prev_opp_move,prev_my_move,prev_outcome,prev2_opp_move,prev2_my_move,prev2_outcome,next_opp_move
2,1,3,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,0,К,Н,lose,Н,Б,lose,Н
3,1,4,Неизвестен,unknown,-1,-0.01,25,Н,Н,1,...,0,2,1,Н,Н,draw,К,Н,lose,Н
4,1,5,Неизвестен,unknown,-1,-0.01,25,Н,К,1,...,0,2,2,Н,Н,draw,Н,Н,draw,Н
8,2,3,Неизвестен,unknown,-1,-0.01,25,Б,Н,0,...,1,0,1,Б,Б,draw,Н,К,win,Б
9,2,4,Неизвестен,unknown,-1,-0.01,25,Б,К,1,...,2,0,0,Б,Н,win,Б,Б,draw,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1196,188,3,Дальний Тайник,<=5,3,0.75,100,Н,К,0,...,0,1,1,Б,Б,draw,Б,К,lose,К
1197,188,4,Дальний Тайник,<=5,3,0.75,100,К,Н,0,...,1,1,0,Н,К,win,Б,Б,draw,Н
1198,188,5,Дальний Тайник,<=5,3,0.75,100,Н,Н,1,...,1,2,0,К,Н,lose,Н,К,win,Б
1199,188,6,Дальний Тайник,<=5,3,0.75,100,Б,Б,1,...,1,2,1,Н,Н,draw,К,Н,lose,Н


In [39]:
X, y = df_ml.drop('next_opp_move', axis=1), df_ml['next_opp_move']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [40]:
drop_features = ["player_name", "match_id"]
cat_features = [ "win_category", "opp_move", "my_move", "outcome", "prev_opp_move", "prev_my_move", "prev_outcome", "prev2_opp_move", "prev2_my_move", "prev2_outcome"]
num_features = ["round", "stake", "opp_match_wins", "opp_match_winrate", "score_me_before", "score_opp_before", "streak_draws", "is_last_round"]
imputer_scaler_encoder = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        ('num', Pipeline([
            ('scaler', RobustScaler())
        ]), num_features),
        ('low_cat', Pipeline([
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preprocessor = Pipeline(
    [
        ('imputer_scaler_encoder', imputer_scaler_encoder)
    ]
)

In [41]:
preprocessor.fit(X_train, y_train)
preprocessed_X_train = preprocessor.transform(X_train)
preprocessed_X_valid = preprocessor.transform(X_valid)

In [42]:
preprocessed_X_train.shape

(510, 40)

In [43]:
# Убедимся, что preprocessed_X_train и preprocessed_X_valid созданы через новый preprocessor
# (это уже должно быть сделано, если вы перезапустили ячейки)

cb = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=3,
    l2_leaf_reg=3,
    bagging_temperature=0.5,
    random_strength=1,
    eval_metric='Accuracy',
    early_stopping_rounds=20,
    verbose=50,
    random_state=42
)

cb.fit(
    preprocessed_X_train, y_train,
    eval_set=(preprocessed_X_valid, y_valid),
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.4196078	test: 0.3906250	best: 0.3906250 (0)	total: 1.07ms	remaining: 212ms
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.4609375
bestIteration = 4

Shrink model to first 5 iterations.


CatBoostClassifier(bagging_temperature=0.5, depth=3, early_stopping_rounds=20, eval_metric='Accuracy', iterations=200, l2_leaf_reg=3, learning_rate=0.1, random_state=42, random_strength=1, verbose=50)

In [45]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- 1. Загрузка данных и обученных объектов ----------
# Предполагается, что df, preprocessor, cb, prob_r2, prob_r3 уже существуют.
# Также нужны df_r2, df_r3 для fallback (если prob_r2/prob_r3 возвращают None).
# Убедитесь, что у вас есть эти объекты.

# ---------- 2. Вспомогательные функции ----------
def counter_move(move):
    """Возвращает контр-ход для заданного хода соперника"""
    if move == 'К': return 'Б'
    if move == 'Н': return 'К'
    if move == 'Б': return 'Н'
    return 'К'  # fallback

def get_move_ml(row_features):
    """Предсказание через ML модель (для раунда >=4)"""
    if isinstance(row_features, dict):
        X = pd.DataFrame([row_features])
    else:
        X = row_features
    X_processed = preprocessor.transform(X)
    pred = cb.predict(X_processed)
    # Извлекаем скаляр
    if isinstance(pred, np.ndarray):
        pred_code = pred.item() if pred.size == 1 else pred[0]
    else:
        pred_code = pred
    # Если pred_code уже строка ('К','Н','Б'), используем напрямую
    if pred_code in ('К', 'Н', 'Б'):
        predicted_opp = pred_code
    else:
        # иначе считаем, что это число 0,1,2
        move_map = {0: 'К', 1: 'Н', 2: 'Б'}
        predicted_opp = move_map[pred_code]
    return counter_move(predicted_opp)

# ---------- 3. Симуляция одного матча ----------
def simulate_match(match_df):
    """
    match_df : DataFrame с одним match_id, отсортированный по round.
    Возвращает: итоговый счёт (score_me - score_opp) и словарь статистики.
    """
    score_me = 0
    score_opp = 0
    n_rounds = len(match_df)
    # Проходим по раундам, но предсказываем ход для следующего раунда
    for i in range(n_rounds):
        curr = match_df.iloc[i]
        rnd = curr['round']
        # Для последнего раунда нет следующего хода — пропускаем
        if i == n_rounds - 1:
            break

        # Получаем реальный ход соперника в следующем раунде
        real_next_opp = match_df.iloc[i+1]['opp_move']

        # Выбираем стратегию в зависимости от номера раунда
        if rnd == 1:
            my_move = get_move_r1(curr['stake'])
        elif rnd == 2:
            # Данные первого раунда
            r1 = match_df.iloc[i-1]
            my_move = get_move_r2(curr['stake'],
                                  r1['outcome'], r1['my_move'], r1['opp_move'],
                                  prob_r2)
            if my_move is None:
                # fallback: использовать get_move_r1
                my_move = get_move_r1(curr['stake'])
        elif rnd == 3:
            # Данные первого и второго раундов
            r1 = match_df.iloc[i-2]
            r2 = match_df.iloc[i-1]
            my_move = get_move_r3(curr['stake'],
                                  r2['outcome'], r2['my_move'], r2['opp_move'],
                                  r1['outcome'], r1['my_move'], r1['opp_move'],
                                  prob_r3)
            if my_move is None:
                # fallback: использовать get_move_r2
                my_move = get_move_r2(curr['stake'],
                                      r2['outcome'], r2['my_move'], r2['opp_move'],
                                      prob_r2)
                if my_move is None:
                    my_move = get_move_r1(curr['stake'])
        else:  # rnd >= 4
            # Признаки для ML берутся из текущего раунда (который уже завершён)
            features = {
                'round': curr['round'],
                'stake': curr['stake'],
                'opp_match_wins': curr['opp_match_wins'],
                'opp_match_winrate': curr['opp_match_winrate'],
                'score_me_before': curr['score_me_before'],
                'score_opp_before': curr['score_opp_before'],
                'streak_draws': curr['streak_draws'],
                'is_last_round': curr['is_last_round'],
                'win_category': curr['win_category'],
                'opp_move': curr['opp_move'],
                'my_move': curr['my_move'],
                'outcome': curr['outcome'],
                'prev_opp_move': curr['prev_opp_move'],
                'prev_my_move': curr['prev_my_move'],
                'prev_outcome': curr['prev_outcome'],
                'prev2_opp_move': curr['prev2_opp_move'],
                'prev2_my_move': curr['prev2_my_move'],
                'prev2_outcome': curr['prev2_outcome']
            }
            my_move = get_move_ml(features)

        # Определяем исход раунда (мой ход vs реальный ход соперника)
        if my_move == real_next_opp:
            outcome = 'draw'
            delta_me, delta_opp = 0, 0
        elif (my_move == 'К' and real_next_opp == 'Н') or \
             (my_move == 'Н' and real_next_opp == 'Б') or \
             (my_move == 'Б' and real_next_opp == 'К'):
            outcome = 'win'
            delta_me, delta_opp = 1, 0
        else:
            outcome = 'loss'
            delta_me, delta_opp = 0, 1

        score_me += delta_me
        score_opp += delta_opp

    return score_me - score_opp, {'score_me': score_me, 'score_opp': score_opp}

# ---------- 4. Прогон по всем матчам ----------
results = []
for match_id, group in df.groupby('match_id'):
    group = group.sort_values('round')
    diff, stats = simulate_match(group)
    results.append({
        'match_id': match_id,
        'diff': diff,
        'score_me': stats['score_me'],
        'score_opp': stats['score_opp']
    })

results_df = pd.DataFrame(results)
print("=== Результаты стратегии (статистика + ML) ===")
print(f"Всего матчей: {len(results_df)}")
print(f"Средняя разница очков: {results_df['diff'].mean():.3f}")
print(f"Общий счёт (me - opp): {results_df['diff'].sum()}")
print(f"Победы (diff>0): {(results_df['diff']>0).sum()}")
print(f"Поражения (diff<0): {(results_df['diff']<0).sum()}")
print(f"Ничьи (diff==0): {(results_df['diff']==0).sum()}")

# ---------- 5. Бейзлайн: всегда камень ----------
def always_rock_simulation(match_df):
    score_me = 0
    score_opp = 0
    for i in range(len(match_df)-1):
        real_next_opp = match_df.iloc[i+1]['opp_move']
        my_move = 'К'
        if my_move == real_next_opp:
            continue
        elif (my_move == 'К' and real_next_opp == 'Н') or \
             (my_move == 'Н' and real_next_opp == 'Б') or \
             (my_move == 'Б' and real_next_opp == 'К'):
            score_me += 1
        else:
            score_opp += 1
    return score_me - score_opp

baseline_diffs = []
for match_id, group in df.groupby('match_id'):
    group = group.sort_values('round')
    diff = always_rock_simulation(group)
    baseline_diffs.append(diff)

print("\n=== Бейзлайн: всегда камень ===")
print(f"Средняя разница очков: {np.mean(baseline_diffs):.3f}")
print(f"Общий счёт: {np.sum(baseline_diffs)}")

=== Результаты стратегии (статистика + ML) ===
Всего матчей: 189
Средняя разница очков: 0.413
Общий счёт (me - opp): 78
Победы (diff>0): 93
Поражения (diff<0): 55
Ничьи (diff==0): 41

=== Бейзлайн: всегда камень ===
Средняя разница очков: 0.000
Общий счёт: 0


In [47]:
import joblib
joblib.dump(cb, 'rps_model.pkl')

['rps_model.pkl']

In [ ]:
# df.to_csv('/content/drive/MyDrive/rps_data.csv', sep=',', index=False)